# Анализ точности PPP-AR
**Сети:** ФАГС (Федеральная активная геодезическая сеть) и IGS (International GNSS Service)

Укажите пути к файлам в ячейке **Настройки** ниже.

In [ ]:
# ── Настройки ────────────────────────────────────────────────────────────────
FAGS_CSV = '/Users/sergeidolin/collaborative-service/fags_batch_report.csv'
IGS_CSV  = '/Users/sergeidolin/collaborative-service/igs_batch_jan2025.csv'

IQR_FACTOR        = 3.0   # коэффициент для отсева выбросов
PER_STATION_FILTER = True  # доп. фильтр выбросов внутри каждого пункта
FIX_Q_VALUES      = {1}   # Q-коды, считаемые «фикс»-решением

# Пороги классов точности (RMS 3D, мм)
GRADE_THR    = [5, 10, 20, 50]
GRADE_LABELS = ['≤5 мм', '5–10 мм', '10–20 мм', '20–50 мм', '>50 мм']

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from IPython.display import display

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Библиотеки загружены.')

In [ ]:
# ── Функции ──────────────────────────────────────────────────────────────────

def load_csv(path, network_name):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    if 'city' not in df.columns:
        df.insert(df.columns.get_loc('station') + 1, 'city', '')
    for col in ('lat','lon','h_m','q','nsat','dN_m','dE_m','dU_m','r3d_m','duration_s'):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['network'] = network_name
    return df

def filter_ok(df):
    mask = (
        (df['status'].str.strip().str.lower() == 'ok')
        & (df['lat'].abs() > 1e-6)
        & df['dN_m'].notna() & df['dE_m'].notna() & df['dU_m'].notna()
        & ~((df['dN_m'].abs() < 1e-9) & (df['dE_m'].abs() < 1e-9) & (df['dU_m'].abs() < 1e-9))
    )
    return df[mask].copy()

def iqr_filter_global(df, factor=IQR_FACTOR):
    mask = pd.Series(True, index=df.index)
    for col in ('dN_m','dE_m','dU_m'):
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        mask &= df[col].between(q1 - factor*iqr, q3 + factor*iqr)
    removed = (~mask).sum()
    return df[mask].copy(), int(removed)

def iqr_filter_per_station(df, factor=IQR_FACTOR):
    keep = []
    for _, grp in df.groupby('station'):
        if len(grp) < 4:
            keep.append(grp); continue
        mask = pd.Series(True, index=grp.index)
        for col in ('dN_m','dE_m','dU_m'):
            q1, q3 = grp[col].quantile(0.25), grp[col].quantile(0.75)
            iqr = q3 - q1
            if iqr < 1e-9: continue
            mask &= grp[col].between(q1 - factor*iqr, q3 + factor*iqr)
        keep.append(grp[mask])
    result = pd.concat(keep)
    return result, len(df) - len(result)

def rms(arr):
    a = np.asarray(arr, dtype=float)
    return float(np.sqrt(np.mean(a**2))) if len(a) else np.nan

def station_stats(df):
    rows = []
    for station, grp in df.groupby('station'):
        city = grp['city'].iloc[0] if grp['city'].iloc[0] else ''
        n = len(grp)
        rn = rms(grp['dN_m'])*1000
        re = rms(grp['dE_m'])*1000
        ru = rms(grp['dU_m'])*1000
        r3 = rms(grp['r3d_m'])*1000 if 'r3d_m' in grp else float(np.sqrt(rn**2+re**2+ru**2))
        fp = 100.0 * grp['q'].isin(FIX_Q_VALUES).sum() / n
        rows.append({'station':station,'city':city,'n':n,
                     'rms_N':rn,'rms_E':re,'rms_U':ru,'rms_3d':r3,'fix_pct':fp})
    return pd.DataFrame(rows).sort_values('station').reset_index(drop=True)

def global_stats(df, st):
    n = len(df)
    return {
        'total_n':    n,
        'n_stations': len(st),
        'fix_global': 100.0*df['q'].isin(FIX_Q_VALUES).sum()/n if n else 0,
        'fix_mean':   st['fix_pct'].mean(),
        'g_N': rms(df['dN_m'])*1000, 'g_E': rms(df['dE_m'])*1000,
        'g_U': rms(df['dU_m'])*1000, 'g_3d': rms(df['r3d_m'])*1000,
        'm_N': st['rms_N'].mean(),   'm_E': st['rms_E'].mean(),
        'm_U': st['rms_U'].mean(),   'm_3d': st['rms_3d'].mean(),
    }

def grade_counts(st):
    vals = st['rms_3d'].values
    prev, result = 0, []
    for thr, lbl in zip(GRADE_THR, GRADE_LABELS[:-1]):
        c = int(np.sum((vals > prev) & (vals <= thr)))
        result.append((lbl, c, 100.0*c/len(vals) if len(vals) else 0))
        prev = thr
    c = int(np.sum(vals > GRADE_THR[-1]))
    result.append((GRADE_LABELS[-1], c, 100.0*c/len(vals) if len(vals) else 0))
    return result

print('Функции определены.')

In [ ]:
# ── Загрузка и фильтрация ─────────────────────────────────────────────────────
df_fags_raw = load_csv(FAGS_CSV, 'ФАГС')
df_igs_raw  = load_csv(IGS_CSV,  'IGS')

def prepare(df_raw, label):
    df_ok = filter_ok(df_raw)
    df, n1 = iqr_filter_global(df_ok)
    n_rem = n1
    if PER_STATION_FILTER:
        df, n2 = iqr_filter_per_station(df)
        n_rem += n2
    st = station_stats(df)
    gs = global_stats(df, st)
    print(f'{label}: строк ok={len(df_ok)}, после фильтра={len(df)} '
          f'(удалено выбросов={n_rem}), пунктов={gs["n_stations"]}')
    return df, st, gs

df_f, st_f, gs_f = prepare(df_fags_raw, 'ФАГС')
df_i, st_i, gs_i = prepare(df_igs_raw,  'IGS')

# Объединённый датасет
df_all = pd.concat([df_f, df_i], ignore_index=True)
st_all = station_stats(df_all)
gs_all = global_stats(df_all, st_all)
print(f'Объединённо: пунктов={gs_all["n_stations"]}, эпох={gs_all["total_n"]}')

---
## Сводная таблица по сетям

In [ ]:
summary_rows = []
for label, gs in [('ФАГС', gs_f), ('IGS', gs_i), ('Объединённо', gs_all)]:
    summary_rows.append({
        'Сеть': label,
        'Пунктов': gs['n_stations'],
        'Эпох': gs['total_n'],
        'Fix% глоб.': f"{gs['fix_global']:.1f}",
        'Fix% ср.': f"{gs['fix_mean']:.1f}",
        'rmsN мм': f"{gs['g_N']:.2f}",
        'rmsE мм': f"{gs['g_E']:.2f}",
        'rmsU мм': f"{gs['g_U']:.2f}",
        'rms3D мм': f"{gs['g_3d']:.2f}",
        'rmsN ср': f"{gs['m_N']:.2f}",
        'rmsE ср': f"{gs['m_E']:.2f}",
        'rmsU ср': f"{gs['m_U']:.2f}",
        'rms3D ср': f"{gs['m_3d']:.2f}",
    })

df_summary = pd.DataFrame(summary_rows).set_index('Сеть')
print('=== Глобальный СКП и доля Fix-решений ===')
display(df_summary)

---
## Классы точности по сетям

In [ ]:
grade_rows = []
for label, st in [('ФАГС', st_f), ('IGS', st_i), ('Объединённо', st_all)]:
    for lbl, cnt, pct in grade_counts(st):
        grade_rows.append({'Сеть': label, 'Класс': lbl, 'Пунктов': cnt, '%': f'{pct:.1f}'})

df_grades = pd.DataFrame(grade_rows).pivot(index='Класс', columns='Сеть', values=['Пунктов','%'])
df_grades = df_grades.reindex(GRADE_LABELS)
print('=== Распределение пунктов по классам точности (RMS 3D) ===')
display(df_grades)

---
## Графики ФАГС

In [ ]:
def plot_residual_histograms(df, gs, title_prefix, color_set=('#2980b9','#e67e22','#27ae60')):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(f'{title_prefix} — Распределение остатков PPP', fontsize=13, fontweight='bold')
    for ax, col, lbl, clr in zip(axes,
            ['dN_m','dE_m','dU_m'], ['dN','dE','dU'], color_set):
        vals = df[col].values * 1000
        ax.hist(vals, bins=60, color=clr, edgecolor='none', alpha=0.85)
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
        ax.axvline(np.mean(vals), color='red', linewidth=1.0, linestyle='-', label=f'μ={np.mean(vals):.1f}')
        ax.set_xlabel(f'{lbl}, мм', fontsize=10)
        ax.set_ylabel('Эпох', fontsize=10)
        ax.set_title(f'СКП = {rms(df[col])*1000:.2f} мм  |  μ = {np.mean(vals):.2f} мм')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

plot_residual_histograms(df_f, gs_f, 'ФАГС')

In [ ]:
def plot_rms3d_stations(st, gs, title_prefix, top_n=60):
    data = st.nlargest(min(top_n, len(st)), 'rms_3d').sort_values('rms_3d', ascending=False)
    clrs = ['#e74c3c' if v > 50 else '#e67e22' if v > 20 else '#f1c40f' if v > 10 else '#27ae60'
            for v in data['rms_3d']]
    fig, ax = plt.subplots(figsize=(max(10, len(data)*0.38), 5))
    bars = ax.bar(data['station'], data['rms_3d'], color=clrs, edgecolor='none', width=0.7)
    ax.axhline(gs['m_3d'], color='navy', linewidth=1.3, linestyle='--',
               label=f'Среднее {gs["m_3d"]:.1f} мм')
    ax.axhline(gs['g_3d'], color='darkred', linewidth=1.0, linestyle=':',
               label=f'Глобальный {gs["g_3d"]:.1f} мм')
    for bar, val in zip(bars, data['rms_3d']):
        if val > gs['m_3d'] * 1.5:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f'{val:.0f}', ha='center', va='bottom', fontsize=7, color='#555')
    ax.set_ylabel('RMS 3D, мм', fontsize=11)
    ax.set_title(f'{title_prefix} — RMS 3D по пунктам (топ-{len(data)})', fontsize=13, fontweight='bold')
    ax.set_xticklabels(data['station'], rotation=90, fontsize=8)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

plot_rms3d_stations(st_f, gs_f, 'ФАГС')

In [ ]:
def plot_rms_neus(st, title_prefix):
    """Столбчатая: rmsN, rmsE, rmsU по пунктам."""
    data = st.sort_values('rms_3d', ascending=False)
    x = np.arange(len(data))
    w = 0.28
    fig, ax = plt.subplots(figsize=(max(12, len(data)*0.4), 5))
    ax.bar(x - w, data['rms_N'], w, label='rmsN', color='#2980b9', alpha=0.85)
    ax.bar(x,     data['rms_E'], w, label='rmsE', color='#e67e22', alpha=0.85)
    ax.bar(x + w, data['rms_U'], w, label='rmsU', color='#27ae60', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(data['station'], rotation=90, fontsize=8)
    ax.set_ylabel('СКП, мм', fontsize=11)
    ax.set_title(f'{title_prefix} — СКП по компонентам N/E/U (сортировка по rms3D)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

plot_rms_neus(st_f, 'ФАГС')

In [ ]:
def plot_fix_rate(st, gs, title_prefix):
    data = st.sort_values('fix_pct')
    clrs = ['#e74c3c' if v < 50 else '#e67e22' if v < 80 else '#27ae60' for v in data['fix_pct']]
    fig, ax = plt.subplots(figsize=(max(10, len(data)*0.38), 4))
    ax.bar(data['station'], data['fix_pct'], color=clrs, edgecolor='none', width=0.7)
    ax.axhline(gs['fix_global'], color='navy', linewidth=1.3, linestyle='--',
               label=f'Глобально {gs["fix_global"]:.1f}%')
    ax.set_ylim(0, 110)
    ax.set_ylabel('Fix-решений, %', fontsize=11)
    ax.set_title(f'{title_prefix} — Доля Fix-решений (Q=1) по пунктам', fontsize=13, fontweight='bold')
    ax.set_xticklabels(data['station'], rotation=90, fontsize=8)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

plot_fix_rate(st_f, gs_f, 'ФАГС')

---
## Графики IGS

In [ ]:
plot_residual_histograms(df_i, gs_i, 'IGS', color_set=('#8e44ad','#c0392b','#16a085'))

In [ ]:
plot_rms3d_stations(st_i, gs_i, 'IGS', top_n=80)

In [ ]:
plot_rms_neus(st_i, 'IGS')

In [ ]:
plot_fix_rate(st_i, gs_i, 'IGS')

---
## Сравнение двух сетей

In [ ]:
# ── Гистограммы остатков ФАГС vs IGS на одном рисунке ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Сравнение остатков PPP: ФАГС vs IGS', fontsize=14, fontweight='bold')

for ax, col, lbl in zip(axes, ['dN_m','dE_m','dU_m'], ['dN','dE','dU']):
    vf = df_f[col].values * 1000
    vi = df_i[col].values * 1000
    # Общий диапазон по шкале IGS (более компактные)
    lim = np.percentile(np.abs(vi), 99) * 1.3
    bins = np.linspace(-lim, lim, 70)
    ax.hist(vi, bins=bins, color='#3498db', alpha=0.7, label=f'IGS  СКП={rms(df_i[col])*1000:.1f}мм', density=True)
    ax.hist(vf, bins=bins, color='#e74c3c', alpha=0.6, label=f'ФАГС СКП={rms(df_f[col])*1000:.1f}мм', density=True)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel(f'{lbl}, мм', fontsize=10)
    ax.set_ylabel('Плотность', fontsize=10)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Сравнение классов точности ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors5 = ['#27ae60','#2ecc71','#f39c12','#e67e22','#e74c3c']

for ax, (label, st) in zip(axes, [('ФАГС', st_f), ('IGS', st_i), ('Объединённо', st_all)]):
    gdata = grade_counts(st)
    lbls  = [g[0] for g in gdata]
    cnts  = [g[1] for g in gdata]
    pcts  = [g[2] for g in gdata]
    bars  = ax.bar(lbls, cnts, color=colors5, edgecolor='none')
    for bar, cnt, pct in zip(bars, cnts, pcts):
        if cnt:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                    f'{cnt}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=8)
    ax.set_title(f'{label}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Пунктов', fontsize=10)
    ax.set_xticklabels(lbls, rotation=20, ha='right', fontsize=9)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

fig.suptitle('Классы точности RMS 3D по пунктам', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Сравнение RMS по компонентам (бар-чарт) ──────────────────────────────────
labels_net = ['ФАГС', 'IGS', 'Объединённо']
gs_list    = [gs_f, gs_i, gs_all]

# Глобальный СКП
global_vals = {
    'N':  [gs['g_N']  for gs in gs_list],
    'E':  [gs['g_E']  for gs in gs_list],
    'U':  [gs['g_U']  for gs in gs_list],
    '3D': [gs['g_3d'] for gs in gs_list],
}
# Средний по пунктам
mean_vals = {
    'N':  [gs['m_N']  for gs in gs_list],
    'E':  [gs['m_E']  for gs in gs_list],
    'U':  [gs['m_U']  for gs in gs_list],
    '3D': [gs['m_3d'] for gs in gs_list],
}

components  = ['N', 'E', 'U', '3D']
comp_colors = ['#2980b9','#e67e22','#27ae60','#8e44ad']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(labels_net))
w = 0.18

for ax, vals_dict, mode in zip(axes,
        [global_vals, mean_vals],
        ['Глобальный СКП', 'Средний по пунктам СКП']):
    for i, (comp, clr) in enumerate(zip(components, comp_colors)):
        ax.bar(x + (i-1.5)*w, vals_dict[comp], w, label=comp, color=clr, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_net, fontsize=11)
    ax.set_ylabel('СКП, мм', fontsize=11)
    ax.set_title(f'{mode} по компонентам N/E/U/3D', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Fix% сравнение ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(labels_net))
fix_g = [gs['fix_global'] for gs in gs_list]
fix_m = [gs['fix_mean']   for gs in gs_list]
w = 0.35
ax.bar(x - w/2, fix_g, w, label='Глобально',        color='#2980b9', alpha=0.85)
ax.bar(x + w/2, fix_m, w, label='Ср. по пунктам',   color='#27ae60', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels_net, fontsize=11)
ax.set_ylim(80, 102)
ax.set_ylabel('Fix-решений, %', fontsize=11)
ax.set_title('Доля Fix-решений (Q=1)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
for xi, vg, vm in zip(x, fix_g, fix_m):
    ax.text(xi - w/2, vg + 0.1, f'{vg:.1f}%', ha='center', va='bottom', fontsize=9)
    ax.text(xi + w/2, vm + 0.1, f'{vm:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## Таблицы СКП по пунктам

In [ ]:
from IPython.display import HTML

def rms3d_color(v):
    if v <= 10:  return '#d5f5e3'
    if v <= 20:  return '#fef9e7'
    if v <= 50:  return '#fdebd0'
    return '#fadbd8'

def show_station_table(st, label):
    df_disp = st[['station','city','n','rms_N','rms_E','rms_U','rms_3d','fix_pct']].copy()
    df_disp.columns = ['Пункт','Город','N эпох','rmsN мм','rmsE мм','rmsU мм','rms3D мм','Fix%']
    for col in ('rmsN мм','rmsE мм','rmsU мм','rms3D мм'):
        df_disp[col] = df_disp[col].round(2)
    df_disp['Fix%'] = df_disp['Fix%'].round(1)

    hdr_style = 'style="background:#2c3e50;color:white;padding:5px 8px;text-align:center"'
    rows_html = []
    for _, row in df_disp.iterrows():
        bg = rms3d_color(row['rms3D мм'])
        cells = ''.join(
            f'<td style="padding:4px 8px;text-align:center;background:{bg}">{row["rms3D мм"]:.2f}</td>'
            if col == 'rms3D мм' else
            f'<td style="padding:4px 8px;text-align:{"left" if col in ("Пункт","Город") else "center"}">{row[col]}</td>'
            for col in df_disp.columns
        )
        rows_html.append(f'<tr>{cells}</tr>')

    headers = ''.join(f'<th {hdr_style}>{c}</th>' for c in df_disp.columns)
    html = f'''<h3>{label} — СКП по пунктам ({len(df_disp)} пунктов)</h3>
<table style="border-collapse:collapse;font-size:13px;font-family:monospace">
  <thead><tr>{headers}</tr></thead>
  <tbody>{''.join(rows_html)}</tbody>
</table>'''
    display(HTML(html))

show_station_table(st_f, 'ФАГС')

In [ ]:
show_station_table(st_i, 'IGS')